# Analyze `minimum_atw` Chunked Output

This notebook inspects the parquet files generated by the chunked antibody-antigen example.

In [ ]:
from pathlib import Path
import json

import pandas as pd

OUT_DIR = Path('/home/eva/minimum_atomworks/out_antibody_antigen_chunked')
OUT_DIR

In [ ]:
tables = {
    'structures': pd.read_parquet(OUT_DIR / 'structures.parquet'),
    'chains': pd.read_parquet(OUT_DIR / 'chains.parquet'),
    'roles': pd.read_parquet(OUT_DIR / 'roles.parquet'),
    'interfaces': pd.read_parquet(OUT_DIR / 'interfaces.parquet'),
    'plugin_status': pd.read_parquet(OUT_DIR / 'plugin_status.parquet'),
    'bad_files': pd.read_parquet(OUT_DIR / 'bad_files.parquet'),
}

summary = pd.DataFrame(
    [
        {
            'table': name,
            'rows': len(df),
            'cols': len(df.columns),
            'columns': ', '.join(df.columns[:12]),
        }
        for name, df in tables.items()
    ]
)
summary

In [ ]:
tables['plugin_status'].groupby(['plugin', 'status'], dropna=False).size().rename('n').reset_index().sort_values(['plugin', 'status'])

In [ ]:
tables['structures'].head()

In [ ]:
tables['roles'][[
    'path',
    'role',
    'id__n_atoms',
    'rolseq__chain_ids',
    'rolseq__n_chains',
    'rolseq__sequence_length_total',
    'rolstat__n_residues',
]].head(10)

In [ ]:
tables['interfaces'][[
    'path',
    'pair',
    'iface__contact_distance',
    'iface__n_contact_atom_pairs',
    'iface__n_left_interface_residues',
    'iface__n_right_interface_residues',
]].head(10)

In [ ]:
analysis_dir = OUT_DIR / 'dataset_analysis'
dataset_annotations = pd.read_parquet(analysis_dir / 'dataset_annotations.parquet')
interface_summary = pd.read_parquet(analysis_dir / 'interface_summary.parquet')
summary_json = json.loads((analysis_dir / 'summary.json').read_text())

dataset_annotations, interface_summary, summary_json

In [ ]:
if not tables['bad_files'].empty:
    display(tables['bad_files'])
else:
    print('No bad files recorded.')